# Rategoan — Panen Korpus (Click-Only)

Notebook ini **mengumpulkan** teks Bahasa Indonesia dari sumber luar (bukan
melatih apa pun — notebook training `colab/latih-gpu.ipynb` tidak disentuh),
lalu memilahnya jadi dua ember berdasarkan lisensi: **terbuka** (aman dipakai
sebagai korpus training setelah direview) dan **tertutup** (disimpan sebagai
arsip/rujukan saja, TIDAK boleh digabung ke korpus training tanpa verifikasi
lisensi manual per artikel).

Hasil panen di-upload ke GitHub Release dengan **tag per sumber**
(`panen-<nama-sumber>`), BUKAN langsung ke tag kanonik `korpus-*-bersih` —
itu baru dilakukan lewat alur review manual sesuai `PRD-DATA-RELEASE.md` §5
(hanya dirigen/Grok yang publish ke tag kanonik).

## DUA MODE

- **Mode A — Whitelist**: ~20 situs individual (JDIH/hukum, GARUDA, Wikisource,
  Gutenberg, portal edukasi pemerintah, berita nasional, blog edukasi, forum
  komunitas). Setiap situs punya default lisensi (pemerintah/Wikimedia/Gutenberg
  = terbuka; berita/blog/forum komersial = tertutup **kecuali** halaman itu
  sendiri secara eksplisit menyatakan lisensi terbuka).
- **Mode B — Dataset aggregator**: Indo4B (CC0), OSCAR id (CC-BY-SA),
  MADLAD-400 id (CC-BY-4.0), CulturaX id — diambil lewat HuggingFace
  `datasets` (streaming, tidak perlu unduh seluruh dataset), difilter
  `lang=id`.

## 2 LANGKAH (yang lain otomatis)

1. Klik ikon kunci (Secrets) di sidebar kiri Colab, tambahkan secret
   **`GITHUB_TOKEN`** (scope `repo`) — cukup sekali. Untuk Mode B, tambahkan
   juga **`HF_TOKEN`** (opsional — hanya dibutuhkan untuk dataset gated
   seperti CulturaX; tanpa ini dataset gated dilewati dengan peringatan,
   bukan error keras).
2. Di sel "Pilih mode" di bawah, pilih **Mode A / Mode B / SEMUA** dari
   dropdown, lalu **Runtime > Run all**.

Sesi Colab mati/putus di tengah panen? Buka lagi notebook ini, ulangi
langkah 2 — progres per sumber otomatis diunduh balik dari Release
(`panen-<sumber>` → asset `progress.json`), URL/dokumen yang sudah diambil
tidak diambil ulang.

In [ ]:
# @title 1. Pilih mode + anggaran panen (satu-satunya pilihan manual)
MODE_PANEN = "SEMUA"  # @param ["Mode A", "Mode B", "SEMUA"]
MAKS_DOKUMEN_PER_SUMBER = 4000  # @param {type:"integer"}
MAKS_HALAMAN_DIKUNJUNGI_PER_SUMBER = 12000  # @param {type:"integer"}
MAKS_DOKUMEN_PER_DATASET = 60000  # @param {type:"integer"}
MAKS_MENIT_PER_DATASET = 12  # @param {type:"integer"}
JEDA_ANTAR_REQUEST_DETIK = 1.5  # @param {type:"number"}

print("Mode:", MODE_PANEN)
print("Anggaran Mode A: maks", MAKS_DOKUMEN_PER_SUMBER, "dokumen /",
      MAKS_HALAMAN_DIKUNJUNGI_PER_SUMBER, "halaman dikunjungi, per sumber.")
print("Anggaran Mode B: maks", MAKS_DOKUMEN_PER_DATASET, "dokumen atau",
      MAKS_MENIT_PER_DATASET, "menit (mana lebih dulu tercapai), per dataset.")
print("Jeda antar request Mode A:", JEDA_ANTAR_REQUEST_DETIK, "detik (sopan ke server sumber).")

In [ ]:
# @title 2. Ambil GITHUB_TOKEN (wajib) + HF_TOKEN (opsional, Mode B gated) dari Colab Secrets
from google.colab import userdata
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
if not GITHUB_TOKEN:
    raise SystemExit(
        'Secret GITHUB_TOKEN belum diset. Klik ikon kunci (Secrets) di sidebar kiri Colab, '
        'tambahkan secret baru bernama GITHUB_TOKEN berisi GitHub Personal Access Token '
        '(scope repo), aktifkan akses notebook ini, lalu Run all lagi.'
    )
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

import os
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
OWNER, REPO = 'maetalizer-png', 'Rategoan'
print('GITHUB_TOKEN OK (panjang', len(GITHUB_TOKEN), 'karakter, tidak ditampilkan).')
print('HF_TOKEN:', 'ADA (dataset gated bisa diakses)' if HF_TOKEN else 'tidak diset (dataset gated akan dilewati)')

In [ ]:
# @title 3. Clone repo (otomatis, token disisipkan di URL - tidak tercetak ke log)
import subprocess, os
REPO_DIR = '/content/Rategoan'
if not os.path.exists(REPO_DIR):
    clone_url = 'https://{}@github.com/{}/{}.git'.format(GITHUB_TOKEN, OWNER, REPO)
    r = subprocess.run(['git', 'clone', '--depth', '1', clone_url, REPO_DIR], capture_output=True, text=True)
    print(r.stdout[-2000:])
    print(r.stderr[-2000:].replace(GITHUB_TOKEN, '***'))
else:
    print('Repo sudah ada di', REPO_DIR)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
# @title 4. Pasang dependensi panen
import subprocess, sys
pkgs = ['trafilatura>=1.8', 'beautifulsoup4', 'requests', 'tqdm', 'datasets>=2.19', 'huggingface_hub']
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, capture_output=True, text=True)
print(r.stdout[-1500:])
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise SystemExit('Gagal pasang dependensi.')
import trafilatura, requests, bs4, datasets as hf_datasets
print('trafilatura', trafilatura.__version__, '| datasets', hf_datasets.__version__, '| OK')

In [ ]:
# @title 5. Filter bahasa Indonesia (fallback, PRD Sec1.1 poin 5) + util lisensi
import re

STOPWORD_ID = set("""
yang dan di ke dari ini itu tidak dengan untuk pada akan adalah dalam oleh
sebagai juga atau karena namun sudah telah bisa dapat para lebih tersebut
tahun kata orang jika saat setelah sebelum antara tentang seperti bagi
""".split())

def rasio_stopword_id(teks):
    kata = re.findall(r"[a-zA-Z]+", teks.lower())
    if len(kata) < 20:
        return 0.0
    cocok = sum(1 for k in kata if k in STOPWORD_ID)
    return cocok / len(kata)

def cukup_indonesia(teks, ambang=0.06):
    """Fallback saja - dipakai kalau sumber tidak punya field lang eksplisit
    (semua sumber Mode A begini). Dihitung dari TEKS YANG SUDAH DIEKSTRAK,
    bukan HTML mentah - sesuai PRD Sec1.1 poin 5."""
    return rasio_stopword_id(teks) >= ambang

MARKER_LISENSI_TERBUKA = [
    (re.compile(r"creativecommons\.org/licenses/([a-z-]+)", re.I), lambda m: "CC-" + m.group(1).upper()),
    (re.compile(r"cc[\s-]?by[\s-]?sa", re.I), lambda m: "CC-BY-SA"),
    (re.compile(r"cc[\s-]?by(?![\s-]?sa)", re.I), lambda m: "CC-BY"),
    (re.compile(r"domain publik|public domain", re.I), lambda m: "Domain Publik"),
    (re.compile(r"lisensi terbuka|open license|open access", re.I), lambda m: "Open Access (terdeteksi halaman)"),
]

def deteksi_lisensi_dari_html(html):
    """Cari penanda lisensi terbuka eksplisit di halaman. Dipakai untuk
    MENAIKKAN status sumber default-tertutup jadi terbuka per-halaman -
    tidak pernah dipakai untuk menurunkan sumber yang sudah default terbuka
    (pemerintah/Wikimedia/Gutenberg)."""
    for pat, fmt in MARKER_LISENSI_TERBUKA:
        m = pat.search(html or "")
        if m:
            return fmt(m)
    return None

print("Filter bahasa + deteksi lisensi siap.")

In [ ]:
# @title 6. WHITELIST Mode A (~20 sumber) - kategori JDIH/GARUDA/Wikisource/Gutenberg/edukasi/berita/blog/forum
WHITELIST_MODE_A = [
    # --- JDIH / produk hukum negara (UU 28/2014 Pasal 41: produk hukum TIDAK
    # berhak cipta, terbuka secara hukum) ---
    dict(slug="jdih-nasional", nama="JDIHN - Jaringan Dok. & Info. Hukum Nasional",
         seed=["https://jdihn.go.id/"], domain="jdihn.go.id",
         lisensi_default="terbuka", label_lisensi="Produk hukum negara (UU 28/2014 psl 41)"),
    dict(slug="jdih-kemenkumham", nama="JDIH Kemenkumham",
         seed=["https://jdih.kemenkumham.go.id/"], domain="jdih.kemenkumham.go.id",
         lisensi_default="terbuka", label_lisensi="Produk hukum negara (UU 28/2014 psl 41)"),
    dict(slug="jdih-setneg", nama="JDIH Sekretariat Negara",
         seed=["https://jdih.setneg.go.id/"], domain="jdih.setneg.go.id",
         lisensi_default="terbuka", label_lisensi="Produk hukum negara (UU 28/2014 psl 41)"),
    dict(slug="jdih-ma", nama="JDIH Mahkamah Agung",
         seed=["https://jdih.mahkamahagung.go.id/"], domain="jdih.mahkamahagung.go.id",
         lisensi_default="terbuka", label_lisensi="Produk hukum negara (UU 28/2014 psl 41)"),
    dict(slug="jdih-dpr", nama="JDIH DPR RI",
         seed=["https://jdih.dpr.go.id/"], domain="jdih.dpr.go.id",
         lisensi_default="terbuka", label_lisensi="Produk hukum negara (UU 28/2014 psl 41)"),
    # --- GARUDA (jurnal ilmiah Indonesia, campuran OA/closed - default
    # tertutup, naik kalau halaman tandai CC/OA eksplisit) ---
    dict(slug="garuda", nama="GARUDA - Garba Rujukan Digital",
         seed=["https://garuda.kemdikbud.go.id/"], domain="garuda.kemdikbud.go.id",
         lisensi_default="tertutup", label_lisensi="Campuran OA/closed - perlu marker CC per artikel"),
    # --- Wikisource & Gutenberg (public domain / CC-BY-SA eksplisit) ---
    dict(slug="wikisource-id", nama="Wikisource Bahasa Indonesia",
         seed=["https://id.wikisource.org/wiki/Halaman_Utama"], domain="id.wikisource.org",
         lisensi_default="terbuka", label_lisensi="CC-BY-SA / Domain Publik (kebijakan Wikimedia)"),
    dict(slug="gutenberg-id", nama="Project Gutenberg (bahasa Indonesia)",
         seed=["https://www.gutenberg.org/browse/languages/id"], domain="www.gutenberg.org",
         lisensi_default="terbuka", label_lisensi="Domain Publik (kebijakan Gutenberg)"),
    # --- Portal edukasi resmi pemerintah (terbuka) ---
    dict(slug="bse-kemdikbud", nama="Buku Sekolah Elektronik Kemdikbud",
         seed=["https://bse.kemdikbud.go.id/"], domain="bse.kemdikbud.go.id",
         lisensi_default="terbuka", label_lisensi="Buku terbuka pemerintah"),
    dict(slug="rumah-belajar", nama="Rumah Belajar Kemdikbud",
         seed=["https://rumahbelajar.kemdikbud.go.id/"], domain="rumahbelajar.kemdikbud.go.id",
         lisensi_default="terbuka", label_lisensi="Materi pendidikan terbuka pemerintah"),
    # --- Berita nasional (default TERTUTUP - hak cipta redaksi, filter lisensi wajib) ---
    dict(slug="kompas", nama="Kompas.com", seed=["https://www.kompas.com/"], domain="www.kompas.com",
         lisensi_default="tertutup", label_lisensi="Hak cipta redaksi - default tertutup"),
    dict(slug="detik", nama="Detik.com", seed=["https://news.detik.com/"], domain="news.detik.com",
         lisensi_default="tertutup", label_lisensi="Hak cipta redaksi - default tertutup"),
    dict(slug="tirto", nama="Tirto.id", seed=["https://tirto.id/"], domain="tirto.id",
         lisensi_default="tertutup", label_lisensi="Hak cipta redaksi - default tertutup"),
    # --- Blog edukasi (default tertutup kecuali marker CC di halaman) ---
    dict(slug="ruangguru-blog", nama="Ruangguru Blog", seed=["https://www.ruangguru.com/blog"],
         domain="www.ruangguru.com", lisensi_default="tertutup", label_lisensi="Blog komersial - cek marker CC"),
    dict(slug="zenius-blog", nama="Zenius Blog", seed=["https://www.zenius.net/blog"],
         domain="www.zenius.net", lisensi_default="tertutup", label_lisensi="Blog komersial - cek marker CC"),
    dict(slug="quipper-blog", nama="Quipper Blog ID", seed=["https://www.quipper.com/id/blog/"],
         domain="www.quipper.com", lisensi_default="tertutup", label_lisensi="Blog komersial - cek marker CC"),
    dict(slug="kompasiana", nama="Kompasiana", seed=["https://www.kompasiana.com/"],
         domain="www.kompasiana.com", lisensi_default="tertutup", label_lisensi="Tulisan komunitas - hak cipta penulis"),
    # --- Forum komunitas (default tertutup, NONAKTIF - ToS forum umumnya
    # melarang scraping otomatis, verifikasi manual dulu sebelum aktifkan) ---
    dict(slug="kaskus", nama="Kaskus Forum", seed=["https://forum.kaskus.co.id/"], domain="forum.kaskus.co.id",
         lisensi_default="tertutup", label_lisensi="Konten pengguna - cek ToS", aktif=False,
         catatan="Nonaktif default: verifikasi ToS forum sebelum aktifkan scraping otomatis."),
    dict(slug="brainly-id", nama="Brainly Indonesia", seed=["https://brainly.co.id/"], domain="brainly.co.id",
         lisensi_default="tertutup", label_lisensi="Konten pengguna - cek ToS", aktif=False,
         catatan="Nonaktif default: verifikasi ToS platform Q&A sebelum aktifkan."),
    dict(slug="reddit-indonesia", nama="Reddit r/indonesia", seed=["https://www.reddit.com/r/indonesia/"],
         domain="www.reddit.com", lisensi_default="tertutup", label_lisensi="ToS API berbayar sejak 2023", aktif=False,
         catatan="Nonaktif default: Reddit API resmi berbayar/berketentuan ketat - jangan scrape HTML langsung."),
]
for s in WHITELIST_MODE_A:
    s.setdefault("aktif", True)
    s.setdefault("catatan", "")

print("Total sumber Mode A terdaftar:", len(WHITELIST_MODE_A))
print("Aktif default:", sum(1 for s in WHITELIST_MODE_A if s["aktif"]),
      "| nonaktif (perlu verifikasi manual):", sum(1 for s in WHITELIST_MODE_A if not s["aktif"]))

In [ ]:
# @title 7. DATASET WHITELIST Mode B (HuggingFace, streaming, lang=id)
WHITELIST_MODE_B = [
    dict(slug="indo4b", nama="Indo4B", hf_id="indobenchmark/indo4b", config=None,
         split="train", field_text="text", lisensi="CC0"),
    dict(slug="oscar-id", nama="OSCAR id", hf_id="oscar-corpus/OSCAR-2301", config="id",
         split="train", field_text="text", lisensi="CC-BY-SA-4.0"),
    dict(slug="madlad400-id", nama="MADLAD-400 id", hf_id="allenai/madlad-400", config="id",
         split="clean", field_text="text", lisensi="CC-BY-4.0"),
    dict(slug="culturax-id", nama="CulturaX id", hf_id="uonlp/CulturaX", config="id",
         split="train", field_text="text", lisensi="Campuran (mC4+OSCAR terkurasi) - lihat kartu dataset", gated=True),
]
for d in WHITELIST_MODE_B:
    d.setdefault("gated", False)

print("Total dataset Mode B:", len(WHITELIST_MODE_B))
print("CATATAN: id HuggingFace di atas adalah rujukan terbaik yang diketahui saat penulisan notebook ini.")
print("Kalau salah satu gagal resolve (dataset dipindah/di-rename di HF), sel 9 akan mencetak")
print("peringatan dan MELEWATI dataset itu, bukan menghentikan seluruh panen - cek HF Hub manual")
print("untuk id pengganti kalau itu terjadi.")

In [ ]:
# @title 8. Util Release (auto-resume + upload) - dipakai kedua mode
import os, json, gzip, time, urllib.request, urllib.error

API = "https://api.github.com/repos/{}/{}".format(OWNER, REPO)

def _api(method, path, token, body=None, timeout=60):
    url = path if path.startswith("http") else API + path
    headers = {"Authorization": "Bearer " + token, "Accept": "application/vnd.github+json"}
    data = None
    if body is not None:
        data = json.dumps(body).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read() or b"{}"
            return resp.status, json.loads(raw)
    except urllib.error.HTTPError as e:
        raw = e.read() or b"{}"
        try:
            parsed = json.loads(raw)
        except Exception:
            parsed = {"raw": raw.decode("utf-8", "replace")}
        return e.code, parsed

def get_or_create_release(tag, title=None, body="Panen korpus staging - lihat manifest.json per sumber."):
    st, rel = _api("GET", "/releases/tags/" + tag, GITHUB_TOKEN)
    if st == 404:
        st, rel = _api("POST", "/releases", GITHUB_TOKEN, {
            "tag_name": tag, "name": title or tag, "body": body, "draft": False, "prerelease": False,
        })
        if st not in (200, 201):
            raise RuntimeError("Gagal buat release {}: {}".format(tag, rel))
    elif st != 200:
        raise RuntimeError("Gagal cek release {}: {}".format(tag, rel))
    return rel

def unduh_asset(tag, nama_asset, tujuan_lokal):
    st, rel = _api("GET", "/releases/tags/" + tag, GITHUB_TOKEN)
    if st != 200:
        return False
    asset = next((a for a in rel.get("assets", []) if a["name"] == nama_asset), None)
    if not asset:
        return False
    req = urllib.request.Request(asset["url"], headers={
        "Authorization": "Bearer " + GITHUB_TOKEN,
        "Accept": "application/octet-stream",
    })
    try:
        with urllib.request.urlopen(req, timeout=120) as resp, open(tujuan_lokal, "wb") as f:
            f.write(resp.read())
        return True
    except Exception as e:
        print("  gagal unduh asset resume", nama_asset, ":", e)
        return False

def unggah_asset(tag, nama_asset, path_lokal, title=None):
    rel = get_or_create_release(tag, title=title)
    existing = next((a for a in rel.get("assets", []) if a["name"] == nama_asset), None)
    if existing:
        _api("DELETE", "/releases/assets/{}".format(existing["id"]), GITHUB_TOKEN)
        _, rel = _api("GET", "/releases/" + str(rel["id"]), GITHUB_TOKEN)
    upload_url = rel["upload_url"].split("{")[0] + "?name=" + nama_asset
    with open(path_lokal, "rb") as f:
        data = f.read()
    req = urllib.request.Request(upload_url, data=data, method="POST", headers={
        "Authorization": "Bearer " + GITHUB_TOKEN,
        "Content-Type": "application/octet-stream",
        "Accept": "application/vnd.github+json",
    })
    with urllib.request.urlopen(req, timeout=1800) as resp:
        return json.loads(resp.read())

def muat_progres(slug, default):
    tag = "panen-" + slug
    lokal = "/content/_progres_{}.json".format(slug)
    if unduh_asset(tag, "progress.json", lokal):
        try:
            with open(lokal, encoding="utf-8") as f:
                progres = json.load(f)
            print("  resume:", slug, "-", progres.get("ringkasan", "progres sebelumnya ditemukan"))
            return progres
        except Exception:
            pass
    return default

def simpan_progres(slug, progres):
    tag = "panen-" + slug
    lokal = "/content/_progres_{}.json".format(slug)
    with open(lokal, "w", encoding="utf-8") as f:
        json.dump(progres, f, ensure_ascii=False)
    try:
        unggah_asset(tag, "progress.json", lokal)
    except Exception as e:
        print("  peringatan: gagal unggah progress.json ke", tag, ":", e)

print("Util Release (auto-resume + upload) siap.")

In [ ]:
# @title 9. Jalankan Mode A - crawler whitelist dengan robots.txt wajib + rate limit
import time, re
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
import requests
from bs4 import BeautifulSoup
import trafilatura

HEADERS = {"User-Agent": "RategoanCorpusBot/1.0 (+https://github.com/maetalizer-png/Rategoan; panen korpus riset bahasa, hormati robots.txt)"}

def robots_boleh(url):
    parsed = urlparse(url)
    robots_url = "{}://{}/robots.txt".format(parsed.scheme, parsed.netloc)
    rp = RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch(HEADERS["User-Agent"], url)
    except Exception:
        return True  # robots.txt tidak terbaca -> anggap boleh, tetap rate-limit sopan

def sama_domain(url, domain):
    try:
        return urlparse(url).netloc.endswith(domain.split("/")[0])
    except Exception:
        return False

def panen_situs(sumber, hasil_terbuka, hasil_tertutup, laporan_sumber):
    slug = sumber["slug"]
    if not sumber.get("aktif", True):
        print("- LEWATI (nonaktif):", sumber["nama"], "-", sumber.get("catatan", ""))
        laporan_sumber[slug] = {"nama": sumber["nama"], "status": "nonaktif", "dokumen": 0}
        return
    if not robots_boleh(sumber["seed"][0]):
        print("- LEWATI (robots.txt melarang):", sumber["nama"])
        laporan_sumber[slug] = {"nama": sumber["nama"], "status": "diblokir robots.txt", "dokumen": 0}
        return

    default_progres = {"dikunjungi": [], "antrian": list(sumber["seed"]), "diambil": 0, "ringkasan": ""}
    progres = muat_progres(slug, default_progres)
    dikunjungi = set(progres["dikunjungi"])
    antrian = list(progres["antrian"]) or list(sumber["seed"])
    diambil = progres["diambil"]

    terbuka_lokal, tertutup_lokal = [], []
    halaman_dicek = 0
    t_mulai = time.time()

    while antrian and diambil < MAKS_DOKUMEN_PER_SUMBER and halaman_dicek < MAKS_HALAMAN_DIKUNJUNGI_PER_SUMBER:
        url = antrian.pop(0)
        if url in dikunjungi:
            continue
        dikunjungi.add(url)
        halaman_dicek += 1
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            time.sleep(JEDA_ANTAR_REQUEST_DETIK)
        except Exception:
            continue
        if resp.status_code != 200 or "text/html" not in resp.headers.get("Content-Type", ""):
            continue

        teks = trafilatura.extract(resp.text, favor_recall=False) or ""
        if len(teks) >= 200 and cukup_indonesia(teks):
            lisensi_terdeteksi = None
            if sumber["lisensi_default"] == "tertutup":
                lisensi_terdeteksi = deteksi_lisensi_dari_html(resp.text)
            lisensi_final = lisensi_terdeteksi or sumber["lisensi_default"]
            rekaman = {
                "text": teks, "source": sumber["nama"], "url": url,
                "license": lisensi_final if lisensi_terdeteksi else sumber["label_lisensi"],
                "lang": "id",
            }
            if lisensi_final == "terbuka" or lisensi_terdeteksi:
                terbuka_lokal.append(rekaman)
            else:
                tertutup_lokal.append(rekaman)
            diambil += 1

        soup = BeautifulSoup(resp.text, "html.parser")
        for a in soup.find_all("a", href=True):
            tautan = urljoin(url, a["href"]).split("#")[0]
            if sama_domain(tautan, sumber["domain"]) and tautan not in dikunjungi and len(antrian) < 4000:
                antrian.append(tautan)

        if diambil and diambil % 200 == 0:
            simpan_progres(slug, {
                "dikunjungi": list(dikunjungi), "antrian": antrian[:2000], "diambil": diambil,
                "ringkasan": "{} dokumen, {} halaman dicek".format(diambil, halaman_dicek),
            })

    simpan_progres(slug, {
        "dikunjungi": list(dikunjungi), "antrian": antrian[:2000], "diambil": diambil,
        "ringkasan": "{} dokumen, {} halaman dicek (selesai sesi ini)".format(diambil, halaman_dicek),
    })
    hasil_terbuka.extend(terbuka_lokal)
    hasil_tertutup.extend(tertutup_lokal)
    laporan_sumber[slug] = {
        "nama": sumber["nama"], "status": "selesai", "dokumen": diambil,
        "terbuka": len(terbuka_lokal), "tertutup": len(tertutup_lokal),
        "halamanDicek": halaman_dicek, "detikDipakai": round(time.time() - t_mulai, 1),
    }
    print("- {}: {} dokumen ({} terbuka / {} tertutup), {} halaman dicek, {:.0f}s".format(
        sumber["nama"], diambil, len(terbuka_lokal), len(tertutup_lokal), halaman_dicek, time.time() - t_mulai))


hasil_terbuka_a, hasil_tertutup_a, laporan_sumber_a = [], [], {}
if MODE_PANEN in ("Mode A", "SEMUA"):
    print("=== MODE A: whitelist ({} sumber) ===".format(len(WHITELIST_MODE_A)))
    for sumber in WHITELIST_MODE_A:
        panen_situs(sumber, hasil_terbuka_a, hasil_tertutup_a, laporan_sumber_a)
    print("Total Mode A: {} terbuka, {} tertutup".format(len(hasil_terbuka_a), len(hasil_tertutup_a)))
else:
    print("Mode A dilewati (MODE_PANEN =", MODE_PANEN, ")")

In [ ]:
# @title 10. Jalankan Mode B - dataset aggregator HuggingFace (streaming, lang=id)
import time
from datasets import load_dataset

def panen_dataset(d, hasil, laporan_sumber):
    slug = d["slug"]
    if d.get("gated") and not HF_TOKEN:
        print("- LEWATI (gated, HF_TOKEN tidak diset):", d["nama"])
        laporan_sumber[slug] = {"nama": d["nama"], "status": "dilewati (gated, tanpa HF_TOKEN)", "dokumen": 0}
        return

    progres = muat_progres(slug, {"offset": 0, "diambil": 0, "ringkasan": ""})
    offset = progres["offset"]
    diambil_total = progres["diambil"]

    try:
        ds = load_dataset(d["hf_id"], d["config"], split=d["split"], streaming=True,
                           token=HF_TOKEN if d.get("gated") else None)
    except Exception as e:
        print("- GAGAL memuat dataset", d["nama"], "(", d["hf_id"], "):", e)
        print("  Kemungkinan id/config HF berubah - cek https://huggingface.co/datasets/" + d["hf_id"])
        laporan_sumber[slug] = {"nama": d["nama"], "status": "gagal dimuat: " + str(e)[:200], "dokumen": 0}
        return

    if offset:
        ds = ds.skip(offset)

    t_mulai = time.time()
    diambil_sesi = 0
    for contoh in ds:
        if diambil_sesi >= MAKS_DOKUMEN_PER_DATASET:
            break
        if (time.time() - t_mulai) / 60.0 >= MAKS_MENIT_PER_DATASET:
            break
        teks = (contoh.get(d["field_text"]) or "").strip()
        offset += 1
        if len(teks) < 200:
            continue
        hasil.append({"text": teks, "source": d["nama"], "url": "hf://" + d["hf_id"], "license": d["lisensi"], "lang": "id"})
        diambil_sesi += 1
        diambil_total += 1
        if diambil_sesi % 2000 == 0:
            simpan_progres(slug, {"offset": offset, "diambil": diambil_total,
                                   "ringkasan": "{} dokumen terkumpul".format(diambil_total)})

    simpan_progres(slug, {"offset": offset, "diambil": diambil_total,
                           "ringkasan": "{} dokumen terkumpul (sesi ini selesai)".format(diambil_total)})
    laporan_sumber[slug] = {"nama": d["nama"], "status": "selesai", "dokumen": diambil_sesi,
                             "totalAkumulasi": diambil_total, "detikDipakai": round(time.time() - t_mulai, 1)}
    print("- {}: {} dokumen sesi ini (total akumulasi {}), {:.0f}s".format(
        d["nama"], diambil_sesi, diambil_total, time.time() - t_mulai))


hasil_terbuka_b, laporan_sumber_b = [], {}
if MODE_PANEN in ("Mode B", "SEMUA"):
    print("=== MODE B: dataset aggregator ({} dataset) ===".format(len(WHITELIST_MODE_B)))
    for d in WHITELIST_MODE_B:
        panen_dataset(d, hasil_terbuka_b, laporan_sumber_b)
    print("Total Mode B: {} dokumen (semua CC0/CC-BY - masuk ember terbuka)".format(len(hasil_terbuka_b)))
else:
    print("Mode B dilewati (MODE_PANEN =", MODE_PANEN, ")")

In [ ]:
# @title 11. Gabung + ukur + tulis manifest (format PRD Sec7, field lang wajib)
import json, gzip, hashlib, collections, datetime

def tulis_jsonl_gz(path, records):
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def komposisi_bahasa(records):
    c = collections.Counter(r.get("lang", "id") for r in records)
    total = sum(c.values()) or 1
    return {k: round(v / total, 4) for k, v in c.items()}

def buat_manifest(nama_tag, records, kategori_catatan):
    path_gz = "/content/{}.jsonl.gz".format(nama_tag)
    tulis_jsonl_gz(path_gz, records)
    manifest = {
        "tag": nama_tag,
        "status": "STAGING - belum masuk korpus kanonik, wajib review manual (PRD Sec5)",
        "catatanKategori": kategori_catatan,
        "generatedAt": datetime.datetime.utcnow().strftime("%Y-%m-%d"),
        "totalDokumen": len(records),
        "totalKataApprox": sum(len(r["text"].split()) for r in records),
        "format": "jsonl.gz",
        "fields": ["text", "source", "license", "url", "lang"],
        "komposisiBahasa": komposisi_bahasa(records),
        "sha256": sha256_file(path_gz) if records else None,
    }
    path_manifest = "/content/manifest-{}.json".format(nama_tag)
    with open(path_manifest, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return path_gz, path_manifest, manifest

paket_terbuka = hasil_terbuka_a + hasil_terbuka_b
paket_tertutup = hasil_tertutup_a

hasil_gabungan = {}
if paket_terbuka:
    hasil_gabungan["terbuka"] = buat_manifest(
        "panen-terbuka", paket_terbuka,
        "Lisensi terbuka (pemerintah/Wikimedia/Gutenberg/CC terverifikasi/dataset CC0-CC-BY). "
        "Masih WAJIB direview manual + dedupe lintas-file sebelum digabung ke korpus-<kategori>-bersih (PRD Sec5).")
    print("terbuka:", hasil_gabungan["terbuka"][2]["totalDokumen"], "dokumen ->", hasil_gabungan["terbuka"][0])
if paket_tertutup:
    hasil_gabungan["tertutup"] = buat_manifest(
        "panen-tertutup", paket_tertutup,
        "Lisensi TIDAK terverifikasi terbuka (berita/blog komersial tanpa marker CC). "
        "Arsip/rujukan saja - DILARANG dipakai untuk training tanpa verifikasi lisensi manual per dokumen.")
    print("tertutup:", hasil_gabungan["tertutup"][2]["totalDokumen"], "dokumen ->", hasil_gabungan["tertutup"][0])
if not hasil_gabungan:
    print("Tidak ada dokumen terkumpul sesi ini (cek anggaran/mode di sel 1).")

In [ ]:
# @title 12. Publikasikan ke Release (tag panen-terbuka / panen-tertutup)
for ember, (path_gz, path_manifest, manifest) in hasil_gabungan.items():
    tag = "panen-" + ember
    judul = "Panen Korpus - {} ({} dokumen)".format(ember.upper(), manifest["totalDokumen"])
    print("Upload ke tag", tag, "...")
    up_gz = unggah_asset(tag, os.path.basename(path_gz), path_gz, title=judul)
    print("  data:", up_gz.get("browser_download_url"))
    up_man = unggah_asset(tag, os.path.basename(path_manifest), path_manifest, title=judul)
    print("  manifest:", up_man.get("browser_download_url"))

print()
print("SELESAI. File 'panen-terbuka' boleh diajukan untuk masuk korpus-<kategori>-bersih setelah")
print("review manual (dedupe lintas-file + klasifikasi kategori PRD Sec1). File 'panen-tertutup'")
print("adalah arsip rujukan - JANGAN dipakai untuk training tanpa verifikasi lisensi eksplisit.")

In [ ]:
# @title 13. Laporan akhir panen (ringkasan per sumber)
import json

laporan = {
    "modePanen": MODE_PANEN,
    "modeA": laporan_sumber_a,
    "modeB": laporan_sumber_b,
    "totalTerbuka": len(paket_terbuka) if "paket_terbuka" in dir() else 0,
    "totalTertutup": len(paket_tertutup) if "paket_tertutup" in dir() else 0,
}
print(json.dumps(laporan, ensure_ascii=False, indent=2))

## Catatan jujur

- **Bukan crawler agresif**: satu request per `JEDA_ANTAR_REQUEST_DETIK`
  detik, `robots.txt` dicek dan **wajib** mengizinkan sebelum satu sumber
  disentuh sama sekali (kalau dilarang, sumber dilewati total, bukan
  "dicoba pelan-pelan").
- **Default lisensi konservatif**: hanya situs pemerintah (produk hukum,
  UU 28/2014 Pasal 41), Wikimedia, dan Gutenberg yang default *terbuka*.
  Berita/blog/forum komersial default *tertutup* — hanya naik ke *terbuka*
  kalau halamannya sendiri secara eksplisit menyebut lisensi CC/domain
  publik. Ini sengaja konservatif; false-negative (kelewat masuk ember
  tertutup padahal sebenarnya boleh) jauh lebih aman daripada
  false-positive.
- **Forum komunitas nonaktif secara default** (Kaskus/Brainly/Reddit) —
  ToS platform semacam ini umumnya membatasi scraping otomatis; aktifkan
  manual (`aktif=True` di sel 6) hanya setelah verifikasi ToS per situs.
- **Id dataset HuggingFace Mode B** (Indo4B/OSCAR/MADLAD-400/CulturaX)
  adalah rujukan terbaik yang diketahui saat notebook ini ditulis — kalau
  salah satu berubah/dipindah di HF Hub, sel 10 mencetak peringatan dan
  melewati dataset itu (tidak menghentikan seluruh panen).
- **Belum masuk korpus training**: baik ember *terbuka* maupun *tertutup*
  adalah **staging**, bukan `korpus-<kategori>-bersih`. Menggabungkannya
  ke korpus kanonik tetap butuh langkah manual sesuai `PRD-DATA-RELEASE.md`
  §5 (dedupe lintas-file, klasifikasi kategori K1/K2/K3, hitung `sha256`,
  hanya dirigen/Grok yang publish ke tag kanonik).